Necessary installs

In [ ]:
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -U transformers peft accelerate trl datasets huggingface_hub
!pip install -U -q bitsandbytes>=0.46.1

Clear training

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
import json
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


def HGlogin(HGtoken):
  login(token=HGtoken)
  return()

def Lamma_setup():
  # Load Llama 3.1 8B
  model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

  bnb_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.bfloat16,
      bnb_4bit_use_double_quant=True,
  )

  model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
  tokenizer = AutoTokenizer.from_pretrained(model_id)
  tokenizer.pad_token = tokenizer.eos_token
  return(model, tokenizer)


def training_data_setup(input_path="train_data.json", output_path = "my_syllogisms.jsonl"):
   #Lamma 3.1 is instruct tuned model, so we need to change the json format to fit the instruct structure
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # If your JSON is a single list of dictionaries
    if not isinstance(data, list):
        data = [data]

    with open(output_path, 'w', encoding='utf-8') as f:
        for entry in data:
            # Create the new dictionary structure
            new_entry = {
                "instruction": entry["syllogism"],
                "response": str(entry["validity"]).lower()  # Converts False to "false"
            }
            # Write as JSONL (JSON Lines) which is best for training
            f.write(json.dumps(new_entry) + '\n')

    print(f"Conversion complete! {len(data)} rows saved to {output_path}")


def formatting_prompts_func(example):
    # The Trainer now passes a single example (dict) at a time this is the place where one can play with the training prompt format
    text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are a logic expert specializing in syllogisms.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"For the following syllogism decide whether it is valid or not. You should respond with single word, without change in capitalization or punctuation, either true, if it is valid, or false otherwise. Here start your syllogism: {example['instruction']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['response']}<|eot_id|>"
    )
    return text # Return a single string

def LoRa_configuration(model, formatted_train_path):
  # Load your local file from the sidebar
  #dataset = load_dataset("json", data_files="my_syllogisms.jsonl", split="train")
  dataset = load_dataset("json", data_files=formatted_train_path, split="train")

  # 1. LoRA Configuration
  peft_config = LoraConfig(
      r=16,
      lora_alpha=32,
      target_modules="all-linear",
      bias="none",
      task_type="CAUSAL_LM"
  )

  training_args = SFTConfig(
      output_dir="./syllogism_model_checkpoints",
      max_length=512,
      dataset_text_field="text",
      packing=False,
      per_device_train_batch_size=5,
      gradient_accumulation_steps=4,
      learning_rate=2e-4,
      num_train_epochs=2,
      logging_steps=1,
      fp16=not torch.cuda.is_bf16_supported(),
      bf16=torch.cuda.is_bf16_supported(),
  )

  # 3. Initialize the Trainer
  trainer = SFTTrainer(
      model=model,
      train_dataset=dataset,
      peft_config=peft_config,
      formatting_func=formatting_prompts_func,
      args=training_args,
  )
  return trainer

# Verify we are on the GPU
print(f"Working on GPU: {torch.cuda.get_device_name(0)}")
HGtoken = input('Your HG writing token:')

HGlogin(HGtoken)
model,tokenizer = Lamma_setup()

#loads the training data and formats them
path_to_training_data = "/content/train_data.json"#input('add path to the training data or add the default one: /content/my_syllogisms.jsonl')
train_dataset=load_dataset("json", data_files=path_to_training_data, split="train")
training_data_setup(path_to_training_data, "/content/my_syllogisms.jsonl")

#loads the test data and formats them
path_to_test_data = "/content/test_data_subtask_1.json"#input('add path to the training data or add the default one: /content/test_data_subtask_1.jsonl')
test_dataset = load_dataset("json", data_files=path_to_test_data, split="train")
training_data_setup("test_data_subtask_1.json", "output_test_data.jsonl")

#setup the trainer
trainer = LoRa_configuration(model, "/content/my_syllogisms.jsonl")

#train the model
trainer.train()

#saves the trained weights
trainer.model.save_pretrained("./content/LoRa_weights")
tokenizer.save_pretrained("./content/LoRa_weights")

#stores the trained model
trained_model = trainer.model



Eval (code by Chatgpt, I have not reviewed it yet)

In [ ]:
trained_model.eval()

# load the formatted test set you already created
formatted_test_dataset = load_dataset("json", data_files="output_test_data.jsonl", split="train")

def build_messages(syllogism_text):
    return [
        {"role": "system", "content": "You are a logic expert specializing in syllogisms."},
        {
            "role": "user",
            "content": (
                "For the following syllogism decide whether it is valid or not. "
                "You should respond with single word, without change in capitalization "
                "or punctuation, either true, if it is valid, or false otherwise. "
                f"Here start your syllogism: {syllogism_text}"
            ),
        },
    ]

def generate_label(model, tokenizer, syllogism_text, max_new_tokens=3):
    messages = build_messages(syllogism_text)

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

    if prediction.startswith("true"):
        return "true"
    elif prediction.startswith("false"):
        return "false"
    else:
        return prediction

base_correct = 0
tuned_correct = 0
base_predictions = []
tuned_predictions = []

for i, example in enumerate(formatted_test_dataset):
    syllogism = example["instruction"]
    gold = example["response"].strip().lower()

    # base model = same model, adapter temporarily disabled
    with trained_model.disable_adapter():
        base_pred = generate_label(trained_model, tokenizer, syllogism)

    # tuned model = adapter active
    tuned_pred = generate_label(trained_model, tokenizer, syllogism)

    base_predictions.append(base_pred)
    tuned_predictions.append(tuned_pred)

    if base_pred == gold:
        base_correct += 1
    if tuned_pred == gold:
        tuned_correct += 1

    print(
        f"Example {i}: gold={gold} | base={base_pred} | tuned={tuned_pred}"
    )

n = len(formatted_test_dataset)
base_accuracy = base_correct / n
tuned_accuracy = tuned_correct / n

print("\n--- FINAL RESULTS ---")
print(f"Number of test examples: {n}")
print(f"Base model accuracy:  {base_accuracy:.4f}")
print(f"Tuned model accuracy: {tuned_accuracy:.4f}")
print(f"Improvement:          {tuned_accuracy - base_accuracy:+.4f}")